In [1]:
import yaml
import numpy as np

def normalize_config(obj):
    """Recursively normalize:
    - numeric strings (incl. scientific notation) -> float
    - 'np.pi', 'np.e', 'np.inf' -> numpy constants
    """
    if isinstance(obj, dict):
        return {k: normalize_config(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [normalize_config(v) for v in obj]

    if isinstance(obj, str):
        s = obj.strip()

        # numpy constants
        if s == "np.pi":
            return np.pi
        if s == "np.e":
            return np.e
        if s in ("np.inf", "inf"):
            return np.inf
        if s in ("-np.inf", "-inf"):
            return -np.inf

        # numeric strings (supports "15.0e9", "200.0e6", "3.5E9", etc.)
        try:
            return float(s)
        except ValueError:
            return obj  # keep as string if not numeric

    return obj  # int/float/bool/None stay as-is


with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg = normalize_config(cfg)




In [2]:
# from Engine_config import Engine
from Engine_V3 import Engine
import json
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import json
from pathlib import Path
import sionnautils
from sionnautils.custom_scene import list_scenes, get_scene
# engine = Engine()


GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
scenes = list_scenes()
print(scenes)
# engine = Engine(cfg, scenes[0])

['nyu_tandon']


In [4]:
np.array(cfg["ue"]["initial_orientation"])[:,0]

array([0., 0.])

In [5]:
from collections import defaultdict

rx_fc = cfg["ue"]["rx_fc"]   # [1.5e10, 1.5e10, 3.5e9, 3.5e9]

fc_to_rx_idx = defaultdict(list)
for rx_idx, fc in enumerate(rx_fc):
    fc_to_rx_idx[fc].append(rx_idx)

n_unique_fc = len(fc_to_rx_idx)

engine = Engine(cfg, scenes[0])

print("Number of unique fc:", n_unique_fc)
print("fc -> rx indices:", dict(fc_to_rx_idx))

bbox_lat: [40.69012764197041, 40.699120858029595]
bbox_long: [-73.99156687083165, -73.97970552916836]
address: 5 MetroTech Center, Brooklyn, NY 11201
descr: NYU Tandon campus
2026-01-18 20:51:27 WARN  [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).
no-name-1      itu_marble
ground         itu_concrete
Number of unique fc: 2
fc -> rx indices: {15000000000.0: [0, 1], 3500000000.0: [2, 3]}


In [6]:
hp_list, sinr_db_list = engine.run_from_file(cfg["route"]["file"])
# hp_list

  0%|          | 0/200 [00:00<?, ?it/s]

/workspace/ruibin/fc_project/UE_fc_hopping_v2/RT_v2/Engine_V3.py:197: RuntimeWarning: divide by zero encountered in log10
  return 10.0 * np.log10(linear_values)
100%|██████████| 200/200 [00:36<00:00,  5.51it/s]


In [14]:
np.array(hp_list)

array([[[[[[[2.70238476e-13]],

           [[0.00000000e+00]]]],



         [[[[2.44206825e-12]],

           [[1.11469646e-14]]]],



         [[[[4.78280526e-10]],

           [[1.16020648e-11]]]],



         [[[[0.00000000e+00]],

           [[0.00000000e+00]]]]],




        [[[[[2.90641345e-13]],

           [[0.00000000e+00]]]],



         [[[[3.28575860e-12]],

           [[1.07139605e-14]]]],



         [[[[3.26635746e-10]],

           [[1.42591147e-11]]]],



         [[[[1.67625344e-10]],

           [[0.00000000e+00]]]]]],





       [[[[[[3.06130175e-13]],

           [[0.00000000e+00]]]],



         [[[[3.52405995e-12]],

           [[1.15308171e-14]]]],



         [[[[6.01639683e-10]],

           [[9.46393658e-12]]]],



         [[[[0.00000000e+00]],

           [[0.00000000e+00]]]]],




        [[[[[2.58051501e-13]],

           [[0.00000000e+00]]]],



         [[[[3.33745444e-12]],

           [[1.07417618e-14]]]],



         [[[[6.45037690e-10]],

        

In [15]:
np.array(sinr_db_list)[:, 0, :, 0, 0, 0]  # SINR for TX 0, first antenna, first subcarrier, first time instant

array([[[ 9.30717187e+00],
        [ 1.75585939e+01],
        [ 1.61455455e+01],
        [-3.00000000e+03]],

       [[ 9.84876145e+00],
        [ 1.91127130e+01],
        [ 1.80253796e+01],
        [-3.00000000e+03]],

       [[ 1.01949120e+01],
        [ 1.98079091e+01],
        [ 1.82991942e+01],
        [ 1.94982532e+01]],

       [[ 1.22568314e+01],
        [ 1.99216950e+01],
        [ 1.74981970e+01],
        [ 1.95927939e+01]],

       [[ 1.08965599e+01],
        [ 1.85840483e+01],
        [ 1.68170555e+01],
        [-3.00000000e+03]],

       [[ 1.27305397e+01],
        [ 1.44228456e+01],
        [ 1.53216484e+01],
        [ 3.54143810e+01]],

       [[ 1.21153182e+01],
        [ 1.81038059e+01],
        [ 1.67091376e+01],
        [ 3.65911433e+01]],

       [[ 1.09178293e+01],
        [ 1.79890722e+01],
        [ 1.85104262e+01],
        [-3.00000000e+03]],

       [[ 1.08046107e+01],
        [ 1.77281143e+01],
        [ 1.87522121e+01],
        [ 1.91610885e+01]],

       [[ 